# Regional Expenditure — Gold
Grain: year × geography × particular. Scope: 2021–2025; includes Central Office and Nationwide. Excludes geographic TOTAL, category GRAND TOTAL and particular ALL summaries.

Run all cells in order with one writer and job maximum concurrent runs = 1. Schedule after Silver and dimensions; do not refresh those inputs during this run. Dimension loading belongs to upstream jobs, including CO and NW members. Existing fact IDs remain unchanged; new IDs start at 60001 and continue above the maximum. No source omissions are deleted.

Amounts retain Silver units. Confirm matching units, reporting periods and budget basis before interpreting the obligation-to-appropriation ratio as execution performance. NULL is not zero; zero denominators produce NULL rates. Negative gaps and rates over 100 are retained for review. Codes align labels, not historical boundaries (NIR/BARMM).

For maps, filter dim_regions.geographic_level = 'region'. Reconciliation below uses included detail rows, not source summary totals.

## 1. Prepare sources
Normalize matching keys and map obligation geography. Keep summaries separate from detail.

In [ ]:
%sql
SET TIME ZONE 'UTC';

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_obligations_prepared AS
SELECT
    obligation_id,
    year,
    TRIM(category) AS category,
    TRIM(particular) AS particular,
    CASE UPPER(TRIM(region))
        WHEN 'BARMM' THEN 'BARMM'
        WHEN 'CARAGA' THEN 'XIII'
        WHEN 'CENTRAL OFFICE' THEN 'CO'
        WHEN 'CORDILLERA ADMINISTRATIVE REGION' THEN 'CAR'
        WHEN 'NATIONAL CAPITAL REGION' THEN 'NCR'
        WHEN 'NATIONWIDE' THEN 'NW'
        WHEN 'NEGROS ISLAND REGION' THEN 'NIR'
        WHEN 'REGION I' THEN 'I'
        WHEN 'REGION II' THEN 'II'
        WHEN 'REGION III' THEN 'III'
        WHEN 'REGION IVA' THEN 'IVA'
        WHEN 'REGION IVB' THEN 'IVB'
        WHEN 'REGION V' THEN 'V'
        WHEN 'REGION VI' THEN 'VI'
        WHEN 'REGION VII' THEN 'VII'
        WHEN 'REGION VIII' THEN 'VIII'
        WHEN 'REGION IX' THEN 'IX'
        WHEN 'REGION X' THEN 'X'
        WHEN 'REGION XI' THEN 'XI'
        WHEN 'REGION XII' THEN 'XII'
    END AS region_code,
    expenditure_amount AS obligation_amount
FROM pondolens.dev_silver.regional_obligations
WHERE year BETWEEN 2021 AND 2025
  AND (
      region IS NULL
      OR UPPER(TRIM(region)) <> 'TOTAL'
  );

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_appropriations_prepared AS
SELECT
    gaa_id,
    year,
    TRIM(category) AS category,
    TRIM(particular) AS particular,
    TRIM(region_code) AS region_code,
    appropriation_amount
FROM pondolens.dev_silver.regional_appropriations
WHERE year BETWEEN 2021 AND 2025;

## 2. Validate source fields
Block missing keys and unmapped geography before filtering summary rows.

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Invalid appropriation fields.')
FROM gold_appropriations_prepared
WHERE year IS NULL OR NULLIF(TRIM(category), '') IS NULL
   OR NULLIF(TRIM(particular), '') IS NULL
   OR NULLIF(TRIM(region_code), '') IS NULL;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Invalid obligation fields or unmapped geography.')
FROM gold_obligations_prepared
WHERE year IS NULL OR NULLIF(TRIM(category), '') IS NULL
   OR NULLIF(TRIM(particular), '') IS NULL OR region_code IS NULL;

## 3. Match detail observations
Full outer join preserves appropriation-only and obligation-only records.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_appropriations_normalized AS
SELECT
    year,
    UPPER(TRIM(category)) AS category,
    UPPER(TRIM(particular)) AS particular,
    UPPER(TRIM(region_code)) AS region_code,
    appropriation_amount,
    1 AS source_present
FROM gold_appropriations_prepared
WHERE UPPER(TRIM(particular)) <> 'ALL'
  AND UPPER(TRIM(category)) <> 'GRAND TOTAL';

CREATE OR REPLACE TEMP VIEW gold_obligations_normalized AS
SELECT
    year,
    UPPER(TRIM(category)) AS category,
    UPPER(TRIM(particular)) AS particular,
    UPPER(TRIM(region_code)) AS region_code,
    obligation_amount,
    1 AS source_present
FROM gold_obligations_prepared
WHERE UPPER(TRIM(particular)) <> 'ALL'
  AND UPPER(TRIM(category)) <> 'GRAND TOTAL';

-- Stop if normalization produces duplicate source keys.
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate normalized source keys.')
FROM (
    SELECT 'APPROPRIATION' AS source,
           year, category, particular, region_code
    FROM gold_appropriations_normalized
    GROUP BY year, category, particular, region_code
    HAVING COUNT(*) > 1

    UNION ALL

    SELECT 'OBLIGATION',
           year, category, particular, region_code
    FROM gold_obligations_normalized
    GROUP BY year, category, particular, region_code
    HAVING COUNT(*) > 1
);

CREATE OR REPLACE TEMP VIEW gold_regional_expenditure_prepared AS
SELECT
    COALESCE(a.year, o.year) AS year,
    COALESCE(a.category, o.category) AS category,
    COALESCE(a.particular, o.particular) AS particular,
    COALESCE(a.region_code, o.region_code) AS region_code,
    a.appropriation_amount,
    o.obligation_amount,
    CASE
        WHEN a.source_present = 1 AND o.source_present = 1
            THEN 'MATCHED'
        WHEN a.source_present = 1 THEN 'APPROPRIATION_ONLY'
        ELSE 'OBLIGATION_ONLY'
    END AS match_status
FROM gold_appropriations_normalized a
FULL OUTER JOIN gold_obligations_normalized o
    ON a.year = o.year
   AND a.category = o.category
   AND a.particular = o.particular
   AND a.region_code = o.region_code;

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) > 0, 'Empty appropriation detail batch.')
FROM gold_appropriations_normalized;
SELECT ASSERT_TRUE(COUNT(*) > 0, 'Empty obligation detail batch.')
FROM gold_obligations_normalized;

## 4. Validate and resolve dimensions
Normalized dimension keys must be unique. Missing members block the load.

In [ ]:
%sql
SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT year_id) AND COUNT(*) = COUNT(DISTINCT year),
    'Invalid year dimension keys.')
FROM pondolens.dev_gold.dim_year;

SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT region_id), 'Invalid region IDs.')
FROM pondolens.dev_gold.dim_regions;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Invalid normalized region keys.')
FROM (
    SELECT NULLIF(UPPER(TRIM(region_code)), '') AS code
    FROM pondolens.dev_gold.dim_regions
    GROUP BY NULLIF(UPPER(TRIM(region_code)), '')
    HAVING COUNT(*) > 1 OR code IS NULL
);

SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT particular_id), 'Invalid particular IDs.')
FROM pondolens.dev_gold.dim_particular;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Invalid normalized particular keys.')
FROM (
    SELECT NULLIF(UPPER(TRIM(category)), '') AS category_key,
           NULLIF(UPPER(TRIM(particular)), '') AS particular_key
    FROM pondolens.dev_gold.dim_particular
    GROUP BY NULLIF(UPPER(TRIM(category)), ''), NULLIF(UPPER(TRIM(particular)), '')
    HAVING COUNT(*) > 1 OR category_key IS NULL OR particular_key IS NULL
);

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_regional_expenditure_resolved AS
SELECT y.year_id, r.region_id, p.particular_id, s.*
FROM gold_regional_expenditure_prepared s
LEFT JOIN pondolens.dev_gold.dim_year y ON s.year = y.year
LEFT JOIN pondolens.dev_gold.dim_regions r
    ON s.region_code = UPPER(TRIM(r.region_code))
LEFT JOIN pondolens.dev_gold.dim_particular p
    ON s.category = UPPER(TRIM(p.category))
   AND s.particular = UPPER(TRIM(p.particular));

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) > 0, 'Empty fact batch.')
FROM gold_regional_expenditure_resolved;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Unresolved dimension keys.')
FROM gold_regional_expenditure_resolved
WHERE year_id IS NULL
   OR region_id IS NULL
   OR particular_id IS NULL;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate fact observation keys.')
FROM (
    SELECT year_id, region_id, particular_id
    FROM gold_regional_expenditure_resolved
    GROUP BY year_id, region_id, particular_id
    HAVING COUNT(*) > 1
);

## 5. Reconcile included source amounts
Compare row counts, NULL counts and totals by year for each source.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_expenditure_reconciliation AS
WITH expected AS (
    SELECT 'APPROPRIATION' AS source, year, COUNT(*) AS rows,
           COUNT(appropriation_amount) AS valued_rows, SUM(appropriation_amount) AS amount
    FROM gold_appropriations_normalized GROUP BY year
    UNION ALL
    SELECT 'OBLIGATION', year, COUNT(*), COUNT(obligation_amount), SUM(obligation_amount)
    FROM gold_obligations_normalized GROUP BY year
), actual AS (
    SELECT 'APPROPRIATION' AS source, year, COUNT(*) AS rows,
           COUNT(appropriation_amount) AS valued_rows, SUM(appropriation_amount) AS amount
    FROM gold_regional_expenditure_resolved
    WHERE match_status IN ('MATCHED', 'APPROPRIATION_ONLY') GROUP BY year
    UNION ALL
    SELECT 'OBLIGATION', year, COUNT(*), COUNT(obligation_amount), SUM(obligation_amount)
    FROM gold_regional_expenditure_resolved
    WHERE match_status IN ('MATCHED', 'OBLIGATION_ONLY') GROUP BY year
)
SELECT e.source, e.year, e.rows AS source_rows, a.rows AS resolved_rows,
       e.amount AS source_amount, a.amount AS resolved_amount,
       (e.rows <=> a.rows) AND (e.valued_rows <=> a.valued_rows)
       AND (e.amount <=> a.amount) AS passed
FROM expected e FULL OUTER JOIN actual a
    ON e.source = a.source AND e.year = a.year;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Source reconciliation failed.')
FROM gold_expenditure_reconciliation WHERE NOT passed OR passed IS NULL;

## 6. Create and validate target
Keep the existing schema and IDs. No drop or overwrite.

In [ ]:
%sql
SET TIME ZONE 'UTC';

CREATE TABLE IF NOT EXISTS pondolens.dev_gold.fact_regional_expenditure (
    fact_regional_expenditure_id BIGINT NOT NULL,
    year_id INT NOT NULL,
    region_id INT NOT NULL,
    particular_id INT NOT NULL,
    appropriation_amount DECIMAL(20,2),
    obligation_amount DECIMAL(20,2),
    execution_gap_amount DECIMAL(21,2),
    execution_rate DECIMAL(20,6),
    match_status STRING,
    processed_at TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) > 0, 'Empty fact batch.')
FROM gold_regional_expenditure_resolved;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Unresolved dimension keys.')
FROM gold_regional_expenditure_resolved
WHERE year_id IS NULL OR region_id IS NULL OR particular_id IS NULL;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate source fact keys.')
FROM (
    SELECT year_id, region_id, particular_id
    FROM gold_regional_expenditure_resolved
    GROUP BY year_id, region_id, particular_id
    HAVING COUNT(*) > 1
);

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT fact_regional_expenditure_id)
    AND COALESCE(COUNT_IF(fact_regional_expenditure_id <= 60000), 0) = 0,
    'Invalid or duplicate target IDs.'
)
FROM pondolens.dev_gold.fact_regional_expenditure;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate target fact keys.')
FROM (
    SELECT year_id, region_id, particular_id
    FROM pondolens.dev_gold.fact_regional_expenditure
    GROUP BY year_id, region_id, particular_id
    HAVING COUNT(*) > 1
);

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Target has unresolved dimension references.')
FROM pondolens.dev_gold.fact_regional_expenditure t
LEFT JOIN pondolens.dev_gold.dim_year y ON t.year_id = y.year_id
LEFT JOIN pondolens.dev_gold.dim_regions r ON t.region_id = r.region_id
LEFT JOIN pondolens.dev_gold.dim_particular p ON t.particular_id = p.particular_id
WHERE y.year_id IS NULL OR r.region_id IS NULL OR p.particular_id IS NULL;

## 7. Assign IDs and calculate measures
Existing observations retain IDs. New observations use maximum ID + row number.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_regional_expenditure_upsert AS
SELECT
    COALESCE(
        t.fact_regional_expenditure_id,
        m.max_id + ROW_NUMBER() OVER (
            PARTITION BY t.fact_regional_expenditure_id IS NULL
            ORDER BY s.year_id, s.region_id, s.particular_id
        )
    ) AS fact_regional_expenditure_id,
    s.year_id,
    s.region_id,
    s.particular_id,
    s.appropriation_amount,
    s.obligation_amount,
    CAST(
        s.appropriation_amount - s.obligation_amount
        AS DECIMAL(21,2)
    ) AS execution_gap_amount,
    CAST(
        (s.obligation_amount / NULLIF(s.appropriation_amount, 0)) * 100
        AS DECIMAL(20,6)
    ) AS execution_rate,
    s.match_status
FROM gold_regional_expenditure_resolved s
LEFT JOIN pondolens.dev_gold.fact_regional_expenditure t
    ON s.year_id = t.year_id
   AND s.region_id = t.region_id
   AND s.particular_id = t.particular_id
CROSS JOIN (
    SELECT COALESCE(MAX(fact_regional_expenditure_id), 60000) AS max_id
    FROM pondolens.dev_gold.fact_regional_expenditure
) m;

## 8. Merge
Update changed measures; insert new observations. Preserve IDs and unchanged timestamps.

In [ ]:
%sql
MERGE INTO pondolens.dev_gold.fact_regional_expenditure t
USING gold_regional_expenditure_upsert s
ON t.year_id = s.year_id
   AND t.region_id = s.region_id
   AND t.particular_id = s.particular_id

WHEN MATCHED AND (
       NOT (t.appropriation_amount <=> s.appropriation_amount)
    OR NOT (t.obligation_amount <=> s.obligation_amount)
    OR NOT (t.execution_gap_amount <=> s.execution_gap_amount)
    OR NOT (t.execution_rate <=> s.execution_rate)
    OR NOT (t.match_status <=> s.match_status)
)
THEN UPDATE SET
    t.appropriation_amount = s.appropriation_amount,
    t.obligation_amount = s.obligation_amount,
    t.execution_gap_amount = s.execution_gap_amount,
    t.execution_rate = s.execution_rate,
    t.match_status = s.match_status,
    t.processed_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    fact_regional_expenditure_id,
    year_id, region_id, particular_id,
    appropriation_amount, obligation_amount,
    execution_gap_amount, execution_rate,
    match_status, processed_at
)
VALUES (
    s.fact_regional_expenditure_id,
    s.year_id, s.region_id, s.particular_id,
    s.appropriation_amount, s.obligation_amount,
    s.execution_gap_amount, s.execution_rate,
    s.match_status, CURRENT_TIMESTAMP()
);

## 9. Verify load
Reconcile every current observation, measures and unique IDs. Target may retain older observations omitted upstream.

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT fact_regional_expenditure_id),
    'Duplicate fact IDs after MERGE.')
FROM pondolens.dev_gold.fact_regional_expenditure;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate fact keys after MERGE.')
FROM (
    SELECT year_id, region_id, particular_id
    FROM pondolens.dev_gold.fact_regional_expenditure
    GROUP BY year_id, region_id, particular_id HAVING COUNT(*) > 1
);

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Post-MERGE reconciliation failed.')
FROM gold_regional_expenditure_resolved s
LEFT JOIN pondolens.dev_gold.fact_regional_expenditure t
    ON s.year_id = t.year_id AND s.region_id = t.region_id
   AND s.particular_id = t.particular_id
WHERE t.fact_regional_expenditure_id IS NULL
   OR NOT (s.appropriation_amount <=> t.appropriation_amount)
   OR NOT (s.obligation_amount <=> t.obligation_amount)
   OR NOT (s.match_status <=> t.match_status)
   OR NOT (CAST(s.appropriation_amount - s.obligation_amount AS DECIMAL(21,2))
           <=> t.execution_gap_amount)
   OR NOT (CAST((s.obligation_amount / NULLIF(s.appropriation_amount, 0)) * 100
           AS DECIMAL(20,6)) <=> t.execution_rate)
   OR t.processed_at IS NULL;

SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT fact_regional_expenditure_id) AS unique_ids,
       MIN(fact_regional_expenditure_id) AS first_id,
       MAX(fact_regional_expenditure_id) AS last_id
FROM pondolens.dev_gold.fact_regional_expenditure;

SELECT year, match_status, COUNT(*) AS row_count
FROM gold_regional_expenditure_resolved
GROUP BY year, match_status ORDER BY year, match_status;

## Rerun check
Run the complete notebook twice with unchanged inputs. The second MERGE must report zero inserts, updates and deletes; IDs and processed_at must remain unchanged. With the currently validated inputs, expect 4487 rows and IDs 60001–64487. New observations continue above the existing maximum. These counts are a baseline, not a permanent constraint.